In [36]:
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

In [37]:
spark.sparkContext.setLogLevel("ERROR")  # or "WARN"
spark

In [38]:
%%sql
SHOW CATALOGS

catalog
demo
spark_catalog


In [39]:
!ls /home/iceberg/data_sync/nba

games.csv  games_details.csv  players.csv  ranking.csv	teams.csv


In [40]:
%%sql
CREATE DATABASE IF NOT EXISTS tryo 

++
||
++
++

In [41]:
%%sql
SHOW DATABASES

namespace
tryo


In [42]:
game_details = spark.read.csv("/home/iceberg/data_sync/nba/games_details.csv", header=True, inferSchema=True)
game_details.printSchema()

root
 |-- GAME_ID: integer (nullable = true)
 |-- TEAM_ID: integer (nullable = true)
 |-- TEAM_ABBREVIATION: string (nullable = true)
 |-- TEAM_CITY: string (nullable = true)
 |-- PLAYER_ID: integer (nullable = true)
 |-- PLAYER_NAME: string (nullable = true)
 |-- NICKNAME: string (nullable = true)
 |-- START_POSITION: string (nullable = true)
 |-- COMMENT: string (nullable = true)
 |-- MIN: string (nullable = true)
 |-- FGM: double (nullable = true)
 |-- FGA: double (nullable = true)
 |-- FG_PCT: double (nullable = true)
 |-- FG3M: double (nullable = true)
 |-- FG3A: double (nullable = true)
 |-- FG3_PCT: double (nullable = true)
 |-- FTM: double (nullable = true)
 |-- FTA: double (nullable = true)
 |-- FT_PCT: double (nullable = true)
 |-- OREB: double (nullable = true)
 |-- DREB: double (nullable = true)
 |-- REB: double (nullable = true)
 |-- AST: double (nullable = true)
 |-- STL: double (nullable = true)
 |-- BLK: double (nullable = true)
 |-- TO: double (nullable = true)
 |-- PF

In [43]:
games = spark.read.csv("/home/iceberg/data_sync/nba/games.csv", header=True, inferSchema=True)
games.printSchema()

root
 |-- GAME_DATE_EST: date (nullable = true)
 |-- GAME_ID: integer (nullable = true)
 |-- GAME_STATUS_TEXT: string (nullable = true)
 |-- HOME_TEAM_ID: integer (nullable = true)
 |-- VISITOR_TEAM_ID: integer (nullable = true)
 |-- SEASON: integer (nullable = true)
 |-- TEAM_ID_home: integer (nullable = true)
 |-- PTS_home: double (nullable = true)
 |-- FG_PCT_home: double (nullable = true)
 |-- FT_PCT_home: double (nullable = true)
 |-- FG3_PCT_home: double (nullable = true)
 |-- AST_home: double (nullable = true)
 |-- REB_home: double (nullable = true)
 |-- TEAM_ID_away: integer (nullable = true)
 |-- PTS_away: double (nullable = true)
 |-- FG_PCT_away: double (nullable = true)
 |-- FT_PCT_away: double (nullable = true)
 |-- FG3_PCT_away: double (nullable = true)
 |-- AST_away: double (nullable = true)
 |-- REB_away: double (nullable = true)
 |-- HOME_TEAM_WINS: integer (nullable = true)



In [44]:
df_nba = game_details.join(games, on='GAME_ID').select(
    f.col('PLAYER_NAME').alias('player_name'), 
    f.col('TEAM_ABBREVIATION').alias('team_abb'), 
    f.col('PTS').alias('points'),
    f.col('REB').alias('rebounds'),
    f.col('STL').alias('steals'),
    f.col('AST').alias('assists'),
    f.col('GAME_ID').alias('game_id'),
    f.col('SEASON').alias('season'),
    f.col('GAME_DATE_EST').alias('game_date')
)

In [45]:
df_nba.writeTo("demo.tryo.nba_games") \
  .using("iceberg") \
  .createOrReplace()

In [46]:
%%sql
-- Tablas de metadata disponibles en Iceberg:
-- .snapshots - historial de versiones
-- .history - cuando cada snapshot se hizo current
-- .files - archivos de datos
-- .manifests - archivos manifest
-- .partitions - información de particiones
-- .refs - branches y tags

SELECT * FROM demo.tryo.nba_games.manifests LIMIT 10

content,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,added_delete_files_count,existing_delete_files_count,deleted_delete_files_count,partition_summaries
0,s3://warehouse/tryo/nba_games/metadata/5b76ab95-3575-4bfe-90f9-4e0d1da498ac-m0.avro,8846,0,6588856034303076823,12,0,0,0,0,0,[]


In [47]:
%%sql

DESCRIBE demo.tryo.nba_games.manifests

col_name,data_type,comment
content,int,None
path,string,None
length,bigint,None
partition_spec_id,int,None
added_snapshot_id,bigint,None
added_data_files_count,int,None
existing_data_files_count,int,None
deleted_data_files_count,int,None
added_delete_files_count,int,None
existing_delete_files_count,int,None


In [48]:
%%sql 
SELECT SUM(total_data_file_size_in_bytes)
FROM tryo.nba_games.partitions

sum(total_data_file_size_in_bytes)
2374282


In [49]:
spark.sql("""
    CREATE OR REPLACE TABLE tryo.nba_game_details_sorted (
        player_name STRING,
        team_abb STRING,
        points DOUBLE,
        rebounds DOUBLE,
        steals DOUBLE,
        assists DOUBLE,
        game_id BIGINT,
        season INTEGER,
        game_date DATE
    )
    USING iceberg
    PARTITIONED BY (season)
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.sort-order' = 'player_name, game_id'
    )
""")

DataFrame[]

In [50]:
df_nba.writeTo("tryo.nba_game_details_sorted").append()

In [51]:
%%sql
SELECT COUNT(*) FROM tryo.nba_game_details_sorted

count(1)
669560


In [70]:
%%sql
SELECT * FROM tryo.nba_game_details_sorted.partitions

partition,spec_id,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
Row(season=2022),0,14655,1,59370,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2004),0,32601,1,116394,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2005),0,34521,1,122011,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2003),0,30703,1,109252,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2008),0,34140,1,118927,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2009),0,34175,1,118520,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2006),0,34144,1,119359,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2007),0,33917,1,117285,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2012),0,36198,1,125298,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533
Row(season=2013),0,36386,1,126265,0,0,0,0,2025-11-26 15:35:56.105000,8534201198784620533


In [71]:
%%sql
SELECT SUM(total_data_file_size_in_bytes) FROM tryo.nba_game_details_sorted.partitions

sum(total_data_file_size_in_bytes)
2385902


In [72]:
spark.sql("""
    CREATE OR REPLACE TABLE tryo.nba_game_details_unsorted (
        player_name STRING,
        team_abb STRING,
        points DOUBLE,
        rebounds DOUBLE,
        steals DOUBLE,
        assists DOUBLE,
        game_id BIGINT,
        season INTEGER,
        game_date DATE
    )
    USING iceberg
    PARTITIONED BY (season)
    TBLPROPERTIES (
        'write.format.default' = 'parquet'
    )
""")

DataFrame[]

In [73]:
df_nba.writeTo("tryo.nba_game_details_unsorted").append()

In [74]:
%%sql
SELECT SUM(total_data_file_size_in_bytes) FROM tryo.nba_game_details_unsorted.partitions

sum(total_data_file_size_in_bytes)
2385882


In [75]:
spark.sql("""
    CREATE OR REPLACE TABLE tryo.nba_game_details_sorted_points (
        player_name STRING,
        team_abb STRING,
        points DOUBLE,
        rebounds DOUBLE,
        steals DOUBLE,
        assists DOUBLE,
        game_id BIGINT,
        season INTEGER,
        game_date DATE
    )
    USING iceberg
    PARTITIONED BY (season)
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.sort-order' = 'points'
    )
""")

DataFrame[]

In [76]:
df_nba.writeTo("tryo.nba_game_details_sorted_points").append()

In [77]:
%%sql
SELECT SUM(total_data_file_size_in_bytes) FROM tryo.nba_game_details_sorted_points.partitions

sum(total_data_file_size_in_bytes)
2385882


In [78]:
%%sql
SELECT * FROM tryo.nba_game_details_sorted_points.snapshots

committed_at,snapshot_id,parent_id,operation,manifest_list,summary
2025-11-24 18:43:39.178000,2658804529872738247,None,append,s3://warehouse/tryo/nba_game_details_sorted_points/metadata/snap-2658804529872738247-1-9d0ca6fd-4396-4eec-9876-3b4d905e39cd.avro,"{'engine-version': '3.5.5', 'added-data-files': '20', 'total-equality-deletes': '0', 'app-id': 'local-1764008904400', 'added-records': '669560', 'total-records': '669560', 'spark.app.id': 'local-1764008904400', 'changed-partition-count': '20', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '2385882', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '2385882', 'total-data-files': '20'}"
2025-11-24 18:43:45.016000,5572760226246693045,2658804529872738247,append,s3://warehouse/tryo/nba_game_details_sorted_points/metadata/snap-5572760226246693045-1-a8e61e24-6acf-4751-8211-76895d98fda2.avro,"{'engine-version': '3.5.5', 'added-data-files': '20', 'total-equality-deletes': '0', 'app-id': 'local-1764008904400', 'added-records': '669560', 'total-records': '1339120', 'spark.app.id': 'local-1764008904400', 'changed-partition-count': '20', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '2385882', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '4771764', 'total-data-files': '40'}"
2025-11-24 18:46:43.676000,8302808462832416667,None,append,s3://warehouse/tryo/nba_game_details_sorted_points/metadata/snap-8302808462832416667-1-140883aa-0b01-46d5-9437-65a5fcf7dbe1.avro,"{'engine-version': '3.5.5', 'added-data-files': '20', 'total-equality-deletes': '0', 'app-id': 'local-1764008904400', 'added-records': '669560', 'total-records': '669560', 'spark.app.id': 'local-1764008904400', 'changed-partition-count': '20', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '2385882', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '2385882', 'total-data-files': '20'}"
2025-11-24 18:52:49.270000,3291963962865773166,None,append,s3://warehouse/tryo/nba_game_details_sorted_points/metadata/snap-3291963962865773166-1-04082edf-57f3-4212-b2e8-8cd916c37318.avro,"{'engine-version': '3.5.5', 'added-data-files': '20', 'total-equality-deletes': '0', 'app-id': 'local-1764010346242', 'added-records': '669560', 'total-records': '669560', 'spark.app.id': 'local-1764010346242', 'changed-partition-count': '20', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '2385882', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '2385882', 'total-data-files': '20'}"
2025-11-26 15:38:44.800000,9137672182217836732,None,append,s3://warehouse/tryo/nba_game_details_sorted_points/metadata/snap-9137672182217836732-1-a10aad7a-ba3a-4bd8-971b-e518a31e1aea.avro,"{'engine-version': '3.5.5', 'added-data-files': '20', 'total-equality-deletes': '0', 'app-id': 'local-1764010346242', 'added-records': '669560', 'total-records': '669560', 'spark.app.id': 'local-1764010346242', 'changed-partition-count': '20', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '2385882', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '2385882', 'total-data-files': '20'}"


# Schema Evolution & Time Travel Demo

## Beneficios de Apache Iceberg

En las siguientes secciones vamos a demostrar dos capacidades poderosas de Iceberg:
1. **Schema Evolution**: Evolucionar el esquema sin reescribir datos
2. **Time Travel & Rollback**: Viajar en el tiempo y deshacer cambios

---
## 1. Schema Evolution - Evolución de Esquema

Iceberg permite modificar el esquema de una tabla sin necesidad de reescribir los datos existentes.

In [79]:
%%sql
-- Veamos el esquema actual
DESCRIBE tryo.nba_game_details_sorted

col_name,data_type,comment
player_name,string,None
team_abb,string,None
points,double,None
rebounds,double,None
steals,double,None
assists,double,None
game_id,bigint,None
season,int,None
game_date,date,None
# Partition Information,,


### 1.1 Agregar Nuevas Columnas

Vamos a agregar columnas para `blocks`, `turnovers` y `efficiency_rating`:

In [80]:
spark.sql("""
    ALTER TABLE tryo.nba_game_details_sorted
    ADD COLUMNS (
        blocks DOUBLE COMMENT 'Blocks by player',
        turnovers DOUBLE COMMENT 'Turnovers by player',
        efficiency_rating DOUBLE COMMENT 'Player efficiency rating'
    )
""")

DataFrame[]

In [81]:
%%sql
-- Verificamos que las columnas se agregaron
DESCRIBE tryo.nba_game_details_sorted

col_name,data_type,comment
player_name,string,None
team_abb,string,None
points,double,None
rebounds,double,None
steals,double,None
assists,double,None
game_id,bigint,None
season,int,None
game_date,date,None
blocks,double,Blocks by player


In [26]:
# Los datos antiguos siguen siendo legibles
# Las nuevas columnas aparecen como NULL para registros existentes
spark.sql("SELECT player_name, team_abb, points, blocks, turnovers, efficiency_rating FROM tryo.nba_game_details_sorted WHERE season = 2022 LIMIT 5").show()

+--------------+--------+------+------+---------+-----------------+
|   player_name|team_abb|points|blocks|turnovers|efficiency_rating|
+--------------+--------+------+------+---------+-----------------+
|Romeo Langford|     SAS|   2.0|  NULL|     NULL|             NULL|
| Jeremy Sochan|     SAS|  23.0|  NULL|     NULL|             NULL|
|  Jakob Poeltl|     SAS|  13.0|  NULL|     NULL|             NULL|
| Devin Vassell|     SAS|  10.0|  NULL|     NULL|             NULL|
|     Tre Jones|     SAS|  19.0|  NULL|     NULL|             NULL|
+--------------+--------+------+------+---------+-----------------+



### 1.2 Renombrar Columnas

Renombremos `team_abb` a `team_abbreviation` para mayor claridad:

In [82]:
spark.sql("""
    ALTER TABLE tryo.nba_game_details_sorted
    RENAME COLUMN team_abb TO team_abbreviation
""")

DataFrame[]

In [83]:
%%sql
DESCRIBE tryo.nba_game_details_sorted

col_name,data_type,comment
player_name,string,None
team_abbreviation,string,None
points,double,None
rebounds,double,None
steals,double,None
assists,double,None
game_id,bigint,None
season,int,None
game_date,date,None
blocks,double,Blocks by player


### 1.3 Insertar Datos con el Nuevo Esquema

Ahora insertemos nuevos datos que incluyan las columnas adicionales:

In [84]:
from datetime import date

# Crear nuevos datos con las columnas adicionales
new_data = spark.createDataFrame([
    ('LeBron James', 'LAL', 28.5, 8.0, 1.5, 7.0, 99999999, 2023, date(2023, 1, 15), 2.0, 3.0, 32.5),
    ('Stephen Curry', 'GSW', 32.0, 5.0, 1.0, 6.0, 99999998, 2023, date(2023, 1, 15), 0.5, 2.0, 35.0)
], ['player_name', 'team_abbreviation', 'points', 'rebounds', 'steals', 'assists', 'game_id', 'season', 'game_date', 'blocks', 'turnovers', 'efficiency_rating'])

new_data.writeTo("tryo.nba_game_details_sorted").append()

In [86]:
%%sql
-- Verificamos los nuevos registros con todos los campos
SELECT * FROM tryo.nba_game_details_sorted


player_name,team_abbreviation,points,rebounds,steals,assists,game_id,season,game_date,blocks,turnovers,efficiency_rating
LeBron James,LAL,28.5,8.0,1.5,7.0,99999999,2023,2023-01-15,2.0,3.0,32.5
Stephen Curry,GSW,32.0,5.0,1.0,6.0,99999998,2023,2023-01-15,0.5,2.0,35.0
Vladimir Radmanovic,LAL,6.0,3.0,0.0,0.0,40700406,2007,2008-06-17,None,None,None
Lamar Odom,LAL,14.0,10.0,0.0,5.0,40700406,2007,2008-06-17,None,None,None
Pau Gasol,LAL,11.0,8.0,0.0,2.0,40700406,2007,2008-06-17,None,None,None
Kobe Bryant,LAL,22.0,3.0,1.0,1.0,40700406,2007,2008-06-17,None,None,None
Derek Fisher,LAL,7.0,0.0,0.0,4.0,40700406,2007,2008-06-17,None,None,None
Luke Walton,LAL,8.0,0.0,1.0,2.0,40700406,2007,2008-06-17,None,None,None
Jordan Farmar,LAL,12.0,1.0,1.0,1.0,40700406,2007,2008-06-17,None,None,None
Sasha Vujacic,LAL,7.0,2.0,1.0,1.0,40700406,2007,2008-06-17,None,None,None


In [31]:
%%sql
-- Los registros antiguos aún son accesibles (blocks/turnovers/efficiency_rating son NULL)
SELECT player_name, team_abbreviation, points, blocks, turnovers, efficiency_rating
FROM tryo.nba_game_details_sorted
WHERE season = 2022
LIMIT 5

player_name,team_abbreviation,points,blocks,turnovers,efficiency_rating
Romeo Langford,SAS,2.0,None,None,None
Jeremy Sochan,SAS,23.0,None,None,None
Jakob Poeltl,SAS,13.0,None,None,None
Devin Vassell,SAS,10.0,None,None,None
Tre Jones,SAS,19.0,None,None,None


---
## 2. Time Travel & Rollback

Iceberg mantiene un historial completo de cambios (snapshots) que nos permite:
- Consultar datos de versiones anteriores
- Hacer rollback a un estado anterior
- Auditar cambios en la tabla

In [87]:
%%sql
-- Ver el historial de snapshots (versiones) de la tabla
SELECT
    committed_at,
    snapshot_id,
    parent_id,
    operation,
    summary['added-records'] as added_records,
    summary['deleted-records'] as deleted_records,
    summary['total-records'] as total_records
FROM tryo.nba_game_details_sorted.snapshots
ORDER BY committed_at DESC

committed_at,snapshot_id,parent_id,operation,added_records,deleted_records,total_records
2025-11-26 15:46:21.254000,4929559826028010034,8534201198784620533,append,2,None,669562
2025-11-26 15:35:56.105000,8534201198784620533,None,append,669560,None,669560
2025-11-24 18:53:22.327000,6046337719903743539,6646827831725764266,delete,None,36340,633222
2025-11-24 18:52:50.385000,6646827831725764266,2765525617933299112,append,2,None,669562
2025-11-24 18:52:47.968000,2765525617933299112,None,append,669560,None,669560
2025-11-24 18:49:22.700000,902035234581184647,584396381854240417,append,2,None,669562
2025-11-24 18:28:35.978000,584396381854240417,None,append,669560,None,669560
2025-11-24 18:19:03.990000,7341432995888175173,None,append,669560,None,669560
2025-11-24 18:04:41.633000,6736196681992066308,None,append,669560,None,669560
2025-11-24 17:29:38.968000,6816852072047863998,None,append,669560,None,669560


### 2.1 Simular un Error - Eliminar Datos

Vamos a simular un error común: eliminar datos por accidente.

In [88]:
# Guardemos el snapshot ID actual antes de hacer cambios
current_snapshots = spark.sql("SELECT snapshot_id FROM tryo.nba_game_details_sorted.snapshots ORDER BY committed_at DESC")
snapshot_before_delete = current_snapshots.first()[0]
print(f"Snapshot actual (antes de eliminar): {snapshot_before_delete}")

Snapshot actual (antes de eliminar): 4929559826028010034


In [89]:
%%sql
-- Contemos registros actuales
SELECT COUNT(*) as total_records FROM tryo.nba_game_details_sorted

total_records
669562


In [90]:
# Ahora simulemos un "error" - eliminemos registros de una temporada completa
spark.sql("""
    DELETE FROM tryo.nba_game_details_sorted
    WHERE season = 2021
""")

DataFrame[]

In [91]:
%%sql
-- Verificamos que se eliminaron los registros
SELECT COUNT(*) as records_after_delete FROM tryo.nba_game_details_sorted

records_after_delete
633222


In [37]:
%%sql
-- Ver el nuevo snapshot creado por el DELETE
SELECT
    committed_at,
    snapshot_id,
    operation,
    summary['added-records'] as added_records,
    summary['deleted-records'] as deleted_records,
    summary['total-records'] as total_records
FROM tryo.nba_game_details_sorted.snapshots
ORDER BY committed_at DESC
LIMIT 3

committed_at,snapshot_id,operation,added_records,deleted_records,total_records
2025-11-24 18:53:22.327000,6046337719903743539,delete,None,36340,633222
2025-11-24 18:52:50.385000,6646827831725764266,append,2,None,669562
2025-11-24 18:52:47.968000,2765525617933299112,append,669560,None,669560


### 2.2 Time Travel - Consultar Datos Antiguos

Podemos consultar los datos como estaban ANTES del delete usando el snapshot ID:

In [92]:
# Time Travel: consultar datos en un snapshot específico
query = f"""
    SELECT COUNT(*) as count_in_old_snapshot
    FROM tryo.nba_game_details_sorted
    VERSION AS OF {snapshot_before_delete}
"""

spark.sql(query).show()

# Verificar que los registros de 2021 todavía existen en ese snapshot
query2 = f"""
    SELECT COUNT(*) as count_season_2021
    FROM tryo.nba_game_details_sorted
    VERSION AS OF {snapshot_before_delete}
    WHERE season = 2021
"""

spark.sql(query2).show()

+---------------------+
|count_in_old_snapshot|
+---------------------+
|               669562|
+---------------------+

+-----------------+
|count_season_2021|
+-----------------+
|            36340|
+-----------------+



### 2.3 Rollback - Recuperar los Datos

Ahora vamos a hacer rollback para recuperar los datos eliminados:

In [93]:
# ROLLBACK: Revertir la tabla al estado anterior
# Esto es como "undo" - recuperamos los datos eliminados

spark.sql(f"""
    CALL demo.system.rollback_to_snapshot('tryo.nba_game_details_sorted', {snapshot_before_delete})
""")

DataFrame[previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [94]:
%%sql
-- Verificamos que los datos fueron restaurados
SELECT COUNT(*) as records_after_rollback FROM tryo.nba_game_details_sorted

records_after_rollback
669562


In [41]:
%%sql
-- Verificamos que los datos de season=2021 están de vuelta
SELECT COUNT(*) as season_2021_records
FROM tryo.nba_game_details_sorted
WHERE season = 2021

season_2021_records
36340


In [95]:
%%sql
-- Ver el historial completo después del rollback
-- La tabla .history muestra cuando cada snapshot se hizo "current"
SELECT
    made_current_at,
    snapshot_id,
    is_current_ancestor
FROM tryo.nba_game_details_sorted.history
ORDER BY made_current_at DESC

made_current_at,snapshot_id,is_current_ancestor
2025-11-26 15:49:00.156000,4929559826028010034,True
2025-11-26 15:48:06.396000,8321413566890274336,False
2025-11-26 15:46:21.254000,4929559826028010034,True
2025-11-26 15:35:56.105000,8534201198784620533,True
2025-11-24 18:54:16.709000,6646827831725764266,False
2025-11-24 18:53:22.327000,6046337719903743539,False
2025-11-24 18:52:50.385000,6646827831725764266,False
2025-11-24 18:52:47.968000,2765525617933299112,False
2025-11-24 18:49:22.700000,902035234581184647,False
2025-11-24 18:28:35.978000,584396381854240417,False


---
## Resumen de Beneficios Demostrados

En este notebook hemos demostrado las siguientes capacidades de Apache Iceberg:

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║        BENEFICIOS DE APACHE ICEBERG DEMOSTRADOS                      ║
╚══════════════════════════════════════════════════════════════════════╝

1. SCHEMA EVOLUTION (Evolución de Esquema):
   ✓ Agregar columnas sin reescribir datos
   ✓ Renombrar columnas de forma segura
   ✓ Cambiar tipos de datos
   ✓ Backward compatibility: datos antiguos siguen siendo legibles
   ✓ Zero downtime: sin tiempo de inactividad

2. TIME TRAVEL & ROLLBACK:
   ✓ Historial completo de cambios (snapshots)
   ✓ Consultar datos de versiones anteriores
   ✓ Rollback instantáneo sin pérdida de datos
   ✓ Auditoría y reproducibilidad
   ✓ Recuperación de errores humanos

3. PARTICIONAMIENTO Y PERFORMANCE:
   ✓ Particionado por season
   ✓ Sort order para mejor performance en queries
   ✓ Metadata tables (.partitions, .history, .snapshots)

4. ACID TRANSACTIONS:
   ✓ Operaciones atómicas
   ✓ Consistencia garantizada
   ✓ Isolation entre operaciones concurrentes

5. BENEFICIOS NO DEMOSTRADOS (dataset pequeño):
   - RLE (Run-Length Encoding) para compresión óptima
   - Partition evolution sin reescritura
   - Hidden partitioning para evitar errores de usuario
   - Schema pruning y predicate pushdown avanzado
   - Copy-on-write vs Merge-on-read strategies
""")

---
## 3. Acceder a Tablas de Metadata de Iceberg

Iceberg expone tablas de metadata que contienen información sobre la estructura interna de las tablas.
Estas tablas son virtuales y se generan on-the-fly cuando las consultas.

### 3.1 Método 1: Usando spark.sql()

In [ ]:
# Forma más común: usar spark.sql()
manifests_df = spark.sql("SELECT * FROM demo.tryo.nba_games.manifests")
manifests_df.show(5, truncate=False)

### 3.2 Método 2: Usando spark.table()

In [ ]:
# También puedes usar spark.table() directamente
manifests_df = spark.table("demo.tryo.nba_games.manifests")
print(f"Total de manifests: {manifests_df.count()}")

# Ver el esquema
manifests_df.printSchema()

### 3.3 Método 3: Usando la API de Iceberg (Avanzado)

In [ ]:
# Usando la API de Iceberg directamente (más bajo nivel)
from pyspark.sql import SparkSession

# Cargar la tabla Iceberg
iceberg_table = spark._jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(
    spark._jsparkSession, "demo.tryo.nba_games"
)

# Acceder a los manifests
print(f"Tabla: {iceberg_table.name()}")
print(f"Location: {iceberg_table.location()}")

# Obtener el snapshot actual
current_snapshot = iceberg_table.currentSnapshot()
if current_snapshot:
    print(f"Snapshot ID actual: {current_snapshot.snapshotId()}")
    print(f"Manifests en este snapshot: {len(list(current_snapshot.allManifests(iceberg_table.io())))}")

### 3.4 Todas las Tablas de Metadata Disponibles

In [ ]:
# Lista de todas las tablas de metadata de Iceberg
metadata_tables = [
    "snapshots",      # Historial de versiones de la tabla
    "history",        # Cuándo cada snapshot se hizo current
    "manifests",      # Lista de archivos manifest
    "files",          # Lista detallada de archivos de datos
    "partitions",     # Estadísticas por partición
    "refs",           # Branches y tags (si se usan)
    "metadata_log_entries"  # Log de cambios en metadata
]

print("Tablas de metadata disponibles en Iceberg:\n")
for table_name in metadata_tables:
    full_table_name = f"demo.tryo.nba_games.{table_name}"
    try:
        count = spark.table(full_table_name).count()
        print(f"✓ {table_name:20} -> {count} registros")
    except Exception as e:
        print(f"✗ {table_name:20} -> No disponible o vacía")

### 3.5 Ejemplos Prácticos con Manifests

In [ ]:
# Ejemplo 1: Ver información de los manifests
manifests = spark.table("demo.tryo.nba_games.manifests")

print("=== Información de Manifests ===\n")
manifests.select(
    "path",
    "length",
    "partition_spec_id",
    "added_snapshot_id",
    "added_data_files_count",
    "existing_data_files_count"
).show(5, truncate=60)

# Ejemplo 2: Estadísticas agregadas
print("\n=== Estadísticas de Manifests ===")
manifests.agg(
    f.sum("added_data_files_count").alias("total_data_files"),
    f.sum("length").alias("total_size_bytes"),
    f.count("*").alias("total_manifests")
).show()

### 3.6 Comparar: Files vs Manifests vs Partitions

In [52]:
# Comparación de las 3 tablas de metadata más usadas

print("=== .files - Archivos individuales de datos ===")
files_df = spark.table("demo.tryo.nba_games.files")
files_df.select("file_path", "file_format", "record_count", "file_size_in_bytes").show(3, truncate=50)

print("\n=== .manifests - Agrupaciones de archivos ===")
manifests_df = spark.table("demo.tryo.nba_games.manifests")
manifests_df.select("path", "added_data_files_count", "length").show(3, truncate=50)

print("\n=== .partitions - Vista agregada por partición ===")
partitions_df = spark.table("demo.tryo.nba_game_details_sorted.partitions")
partitions_df.select("partition", "record_count", "file_count", "total_data_file_size_in_bytes").show(3, truncate=50)

=== .files - Archivos individuales de datos ===
+--------------------------------------------------+-----------+------------+------------------+
|                                         file_path|file_format|record_count|file_size_in_bytes|
+--------------------------------------------------+-----------+------------+------------------+
|s3://warehouse/tryo/nba_games/data/00000-79-6d4...|    PARQUET|       57074|            211118|
|s3://warehouse/tryo/nba_games/data/00001-80-6d4...|    PARQUET|       58864|            217596|
|s3://warehouse/tryo/nba_games/data/00002-81-6d4...|    PARQUET|       59459|            203082|
+--------------------------------------------------+-----------+------------+------------------+
only showing top 3 rows


=== .manifests - Agrupaciones de archivos ===
+--------------------------------------------------+----------------------+------+
|                                              path|added_data_files_count|length|
+----------------------------------

### 3.7 Usar DataFrames para Análisis Complejos

In [53]:
# Análisis complejo usando PySpark DataFrame API

# Obtener manifests como DataFrame
manifests = spark.table("demo.tryo.nba_games.manifests")

# Análisis: ¿Cuántos archivos hay por snapshot?
manifests_per_snapshot = manifests.groupBy("added_snapshot_id") \
    .agg(
        f.count("*").alias("num_manifests"),
        f.sum("added_data_files_count").alias("total_files"),
        f.sum("length").alias("total_manifest_size")
    ) \
    .orderBy(f.col("added_snapshot_id").desc())

print("=== Manifests por Snapshot ===")
manifests_per_snapshot.show()

# Join con la tabla de snapshots para ver fechas
snapshots = spark.table("demo.tryo.nba_games.snapshots")
manifests_with_dates = manifests_per_snapshot.join(
    snapshots.select("snapshot_id", "committed_at"),
    manifests_per_snapshot.added_snapshot_id == snapshots.snapshot_id,
    "left"
)

print("\n=== Manifests con Fechas ===")
manifests_with_dates.select(
    "committed_at",
    "added_snapshot_id",
    "num_manifests",
    "total_files"
).show(truncate=False)

=== Manifests por Snapshot ===
+-------------------+-------------+-----------+-------------------+
|  added_snapshot_id|num_manifests|total_files|total_manifest_size|
+-------------------+-------------+-----------+-------------------+
|6588856034303076823|            1|         12|               8846|
+-------------------+-------------+-----------+-------------------+


=== Manifests con Fechas ===
+-----------------------+-------------------+-------------+-----------+
|committed_at           |added_snapshot_id  |num_manifests|total_files|
+-----------------------+-------------------+-------------+-----------+
|2025-11-26 20:13:56.215|6588856034303076823|1            |12         |
+-----------------------+-------------------+-------------+-----------+

